# Etape 3 - Modelisation fraude (Machine Learning)

Notebook de depart pour predire `fraud_reported` a partir de `insurance_claims.csv`.


In [5]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.tree import DecisionTreeClassifier
from catboost import CatBoostClassifier

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.model_selection import RandomizedSearchCV


## 1



In [6]:
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

# Chargement robuste du dataset
candidate_paths = [
    Path('process/insurance_claims_cleaned.csv'),
    Path('../process/insurance_claims_cleaned.csv'),
    Path('insurance_claims_cleaned.csv')
]
DATA_PATH = next((p for p in candidate_paths if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError('insurance_claims_cleaned.csv introuvable.')

df = pd.read_csv(DATA_PATH).copy()

# Feature engineering minimal (si colonnes absentes)
if 'policy_bind_date' in df.columns and 'incident_date' in df.columns:
    df['policy_bind_date'] = pd.to_datetime(df['policy_bind_date'])
    df['incident_date'] = pd.to_datetime(df['incident_date'])
    if 'claim_delay_days' not in df.columns:
        df['claim_delay_days'] = (df['incident_date'] - df['policy_bind_date']).dt.days
    if 'incident_month' not in df.columns:
        df['incident_month'] = df['incident_date'].dt.month
    if 'policy_year' not in df.columns:
        df['policy_year'] = df['policy_bind_date'].dt.year

if 'rel1_early_incident' not in df.columns:
    low_severity = df['incident_severity'].isin(['MINOR DAMAGE', 'TRIVIAL DAMAGE'])
    df['rel1_early_incident'] = (df['claim_delay_days'] < 30).astype(int)
    df['rel2_severity_amount_mismatch'] = (low_severity & (df['total_claim_amount'] > 70000)).astype(int)
    df['rel3_just_above_deductible'] = ((df['policy_deductable'] < df['total_claim_amount']) & (df['total_claim_amount'] < df['policy_deductable'] + 1000)).astype(int)
    df['rel4_no_police_high_amount'] = ((df['police_report_available'] == 'NO') & (df['total_claim_amount'] > 70000)).astype(int)
    df['rel5_low_premium_high_claim'] = ((df['policy_annual_premium'] < 1100) & (df['total_claim_amount'] > 70000)).astype(int)
    if 'incident_hour_of_the_day' in df.columns:
        is_night = (df['incident_hour_of_the_day'] >= 23) | (df['incident_hour_of_the_day'] <= 5)
        df['rel6_night_single_vehicle'] = (is_night & (df['incident_type'] == 'SINGLE VEHICLE COLLISION')).astype(int)
    else:
        df['rel6_night_single_vehicle'] = 0

# X / y
target = 'fraud_reported'
y = df[target].map({'N': 0, 'Y': 1})

selected_features = [
    'total_claim_amount', 'policy_deductable', 'policy_annual_premium',
    'incident_severity', 'police_report_available', 'incident_type',
    'witnesses', 'bodily_injuries', 'claim_delay_days',
    'incident_month', 'policy_year',
    'rel1_early_incident', 'rel2_severity_amount_mismatch',
    'rel3_just_above_deductible', 'rel4_no_police_high_amount',
    'rel5_low_premium_high_claim', 'rel6_night_single_vehicle',
    'age', 'months_as_customer'
]

X = df[selected_features].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.4, random_state=42, stratify=y
)

num_cols = X.select_dtypes(include=['number']).columns.tolist()
cat_cols = X.select_dtypes(include=['object']).columns.tolist()

preprocess = ColumnTransformer([
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median'))]), num_cols),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ]), cat_cols)
])

print(f'Dataset: {DATA_PATH} | shape={df.shape}')
print(f'Train={X_train.shape} Test={X_test.shape} Fraud ratio={y.mean():.3f}')
print('Setup OK: Pipeline, preprocess, X_train/X_test, y_train/y_test disponibles.')


Dataset: ..\process\insurance_claims_cleaned.csv | shape=(1000, 45)
Train=(600, 19) Test=(400, 19) Fraud ratio=0.247
Setup OK: Pipeline, preprocess, X_train/X_test, y_train/y_test disponibles.


## 2) Chargement des donnees


In [7]:
DATA_PATH = '../process/insurance_claims_cleaned.csv'
df = pd.read_csv(DATA_PATH)
print('Shape:', df.shape)
df.head()


Shape: (1000, 36)


,months_as_customer,age,policy_bind_date,policy_state,policy_csl,policy_deductable,policy_annual_premium,umbrella_limit,insured_sex,insured_education_level,...,witnesses,police_report_available,total_claim_amount,injury_claim,property_claim,vehicle_claim,auto_make,auto_model,auto_year,fraud_reported
0,328,48,2014-10-17,OH,250/500,1000,1406.91,0,MALE,MD,...,2,YES,71610,6510,13020,52080,SAAB,92X,2004,Y
1,228,42,2006-06-27,IN,250/500,2000,1197.22,5000000,MALE,MD,...,0,NO,5070,780,780,3510,MERCEDES,E400,2007,Y
2,134,29,2000-09-06,OH,100/300,2000,1413.14,5000000,FEMALE,PHD,...,3,NO,34650,7700,3850,23100,DODGE,RAM,2007,N
3,256,41,1990-05-25,IL,250/500,2000,1415.74,6000000,FEMALE,PHD,...,2,NO,63400,6340,6340,50720,CHEVROLET,TAHOE,2014,Y
4,228,44,2014-06-06,IL,500/1000,1000,1583.91,6000000,MALE,ASSOCIATE,...,1,NO,6500,1300,650,4550,ACCURA,RSX,2009,N


## 3) Nettoyage minimum pour modelisation


In [8]:
# Preprocessing & Feature Engineering Expert
if '_c39' in df.columns:
    df = df.drop(columns=['_c39'])
df = df.replace('?', np.nan)

# Conversion dates
for c in ['policy_bind_date', 'incident_date']:
    df[c] = pd.to_datetime(df[c], errors='coerce')

# 1. Relation 1 : Early Incident (< 30 jours)
df['claim_delay_days'] = (df['incident_date'] - df['policy_bind_date']).dt.days
df['rel1_early_incident'] = (df['claim_delay_days'] < 30).astype(int)

# Seuils statistiques pour les calculs suivants
high_amount = df['total_claim_amount'].quantile(0.75)
low_premium = df['policy_annual_premium'].quantile(0.25)

# 2. Relation 2 : Gravité faible + montant élevé
low_severity = df['incident_severity'].isin(['Minor Damage', 'Trivial Damage'])
df['rel2_severity_amount_mismatch'] = (low_severity & (df['total_claim_amount'] > high_amount)).astype(int)

# 3. Relation 3 : Proximité franchise (Franchise < Montant < Franchise + 1000)
df['rel3_just_above_deductible'] = ((df['total_claim_amount'] > df['policy_deductable']) & 
                                    (df['total_claim_amount'] < df['policy_deductable'] + 1000)).astype(int)

# 4. Relation 4 : Pas de rapport de police + montant élevé
df['rel4_no_police_high_amount'] = ((df['police_report_available'] == 'NO') & 
                                    (df['total_claim_amount'] > high_amount)).astype(int)

# 5. Relation 5 : Petite prime + énorme réclamation
df['rel5_low_premium_high_claim'] = ((df['policy_annual_premium'] < low_premium) & 
                                     (df['total_claim_amount'] > high_amount)).astype(int)

# 6. Relation 6 : Nuit + Single Vehicle
is_night = df['incident_hour_of_the_day'].between(23, 23) | df['incident_hour_of_the_day'].between(0, 5)
df['rel6_night_single_vehicle'] = (is_night & (df['incident_type'] == 'Single Vehicle Collision')).astype(int)

# Features temporelles classiques de base
df['incident_month'] = df['incident_date'].dt.month
df['policy_year'] = df['policy_bind_date'].dt.year

df.head()


,months_as_customer,age,policy_bind_date,policy_state,policy_csl,policy_deductable,policy_annual_premium,umbrella_limit,insured_sex,insured_education_level,...,fraud_reported,claim_delay_days,rel1_early_incident,rel2_severity_amount_mismatch,rel3_just_above_deductible,rel4_no_police_high_amount,rel5_low_premium_high_claim,rel6_night_single_vehicle,incident_month,policy_year
0,328,48,2014-10-17,OH,250/500,1000,1406.91,0,MALE,MD,...,Y,100,0,0,0,0,0,0,1,2014
1,228,42,2006-06-27,IN,250/500,2000,1197.22,5000000,MALE,MD,...,Y,3130,0,0,0,0,0,0,1,2006
2,134,29,2000-09-06,OH,100/300,2000,1413.14,5000000,FEMALE,PHD,...,N,5282,0,0,0,0,0,0,2,2000
3,256,41,1990-05-25,IL,250/500,2000,1415.74,6000000,FEMALE,PHD,...,Y,8996,0,0,0,0,0,0,1,1990
4,228,44,2014-06-06,IL,500/1000,1000,1583.91,6000000,MALE,ASSOCIATE,...,N,256,0,0,0,0,0,0,2,2014


## 4) Definition X / y + split


In [9]:
target = 'fraud_reported'
y = df[target].map({'N': 0, 'Y': 1})

# On garde vos 10 colonnes + les flags experts + les 3 colonnes boosters
selected_features = [
    # Votre sélection initiale
    'total_claim_amount', 'policy_deductable', 'policy_annual_premium', 
    'incident_severity', 'police_report_available', 'incident_type', 
    'witnesses', 'bodily_injuries', 'claim_delay_days', 
    'incident_month', 'policy_year', 
    # Les relation expertes (Rel 1 à 6)
    'rel1_early_incident', 'rel2_severity_amount_mismatch', 
    'rel3_just_above_deductible', 'rel4_no_police_high_amount', 
    'rel5_low_premium_high_claim', 'rel6_night_single_vehicle',
    # Les colonnes boosters (forte puissance prédictive)
    'age', 'months_as_customer', 'insured_hobbies'
]

X = df[selected_features]

print('Fraud ratio:', y.mean().round(3))
print('Nombre total de features:', len(selected_features))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.4, random_state=42, stratify=y
)

print('Train:', X_train.shape, 'Test:', X_test.shape)


Fraud ratio: 0.247
Nombre total de features: 20
Train: (600, 20) Test: (400, 20)


## 5) Pipeline preprocessing


In [10]:
num_cols = X.select_dtypes(include=['number']).columns.tolist()
cat_cols = X.select_dtypes(exclude=['number']).columns.tolist()

num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
])

cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocess = ColumnTransformer([
    ('num', num_pipe, num_cols),
    ('cat', cat_pipe, cat_cols)
])

print('Numeriques:', len(num_cols), 'Categoriels:', len(cat_cols))


Numeriques: 16 Categoriels: 4


## 6) Baseline 1 - Logistic Regression


In [6]:
logreg_model = Pipeline([
    ('prep', preprocess),
    ('clf', LogisticRegression(max_iter=1000, class_weight='balanced'))
])

logreg_model.fit(X_train, y_train)
pred_lr = logreg_model.predict(X_test)
proba_lr = logreg_model.predict_proba(X_test)[:, 1]

print('=== Logistic Regression ===')
print(classification_report(y_test, pred_lr, digits=3))
print('Confusion matrix:\n', confusion_matrix(y_test, pred_lr))
print('ROC-AUC:', round(roc_auc_score(y_test, proba_lr), 3))


=== Logistic Regression ===
              precision    recall  f1-score   support

           0      0.908     0.854     0.880       301
           1      0.624     0.737     0.676        99

    accuracy                          0.825       400
   macro avg      0.766     0.796     0.778       400
weighted avg      0.838     0.825     0.830       400

Confusion matrix:
 [[257  44]
 [ 26  73]]
ROC-AUC: 0.815


## 7) Baseline 2 - Random Forest


In [11]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

rf_model = Pipeline([
    ('prep', preprocess),
    ('clf', RandomForestClassifier(
        n_estimators=400,
        random_state=42,
        class_weight='balanced_subsample',
        n_jobs=-1
    ))
])

rf_model.fit(X_train, y_train)
pred_rf = rf_model.predict(X_test)
proba_rf = rf_model.predict_proba(X_test)[:, 1]

print('=== Random Forest ===')
print(classification_report(y_test, pred_rf, digits=3))
print('Confusion matrix:\n', confusion_matrix(y_test, pred_rf))
print('ROC-AUC:', round(roc_auc_score(y_test, proba_rf), 3))


=== Random Forest ===
              precision    recall  f1-score   support

           0      0.844     0.884     0.864       301
           1      0.588     0.505     0.543        99

    accuracy                          0.790       400
   macro avg      0.716     0.694     0.704       400
weighted avg      0.781     0.790     0.784       400

Confusion matrix:
 [[266  35]
 [ 49  50]]
ROC-AUC: 0.84


In [12]:
from xgboost import XGBClassifier

# Pipeline XGBoost
xgb_model = Pipeline([
    ('prep', preprocess),
    ('clf', XGBClassifier(
        n_estimators=100,
        max_depth=5,
        learning_rate=0.01,
        scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),  # pour déséquilibre
        use_label_encoder=False,
        eval_metric='logloss',
         # Bonus importants:
        subsample=0.8,      # Échantillonnage des données (0.5-1.0)
        colsample_bytree=0.8,  # Échantillonnage des features
        min_child_weight=1,    # Régularisation
        random_state=42
    ))
])

xgb_model.fit(X_train, y_train)
pred_xgb = xgb_model.predict(X_test)
proba_xgb = xgb_model.predict_proba(X_test)[:, 1]

print('=== XGBoost ===')
print(classification_report(y_test, pred_xgb, digits=3))
print('Confusion matrix:\n', confusion_matrix(y_test, pred_xgb))
print('ROC-AUC:', round(roc_auc_score(y_test, proba_xgb), 3))

=== XGBoost ===
              precision    recall  f1-score   support

           0      0.941     0.841     0.888       301
           1      0.634     0.838     0.722        99

    accuracy                          0.840       400
   macro avg      0.787     0.839     0.805       400
weighted avg      0.865     0.840     0.847       400

Confusion matrix:
 [[253  48]
 [ 16  83]]
ROC-AUC: 0.845


## 7bis) Modèle Arbre - Decision Tree
Nous ajoutons un arbre de décision simple pour illustrer la famille des modèles à base d'arbre.

In [13]:
from sklearn.tree import DecisionTreeClassifier

# Pipeline Decision Tree
dt_model = Pipeline([
    ('prep', preprocess),
    ('clf', DecisionTreeClassifier(
        max_depth=6,
        class_weight='balanced',
        random_state=42
))
])

dt_model.fit(X_train, y_train)
pred_dt = dt_model.predict(X_test)
proba_dt = dt_model.predict_proba(X_test)[:, 1]

print('=== Decision Tree ===')
print(classification_report(y_test, pred_dt, digits=3))
print('Confusion matrix:\n', confusion_matrix(y_test, pred_dt))
print('ROC-AUC:', round(roc_auc_score(y_test, proba_dt), 3))

=== Decision Tree ===
              precision    recall  f1-score   support

           0      0.896     0.857     0.876       301
           1      0.616     0.697     0.654        99

    accuracy                          0.818       400
   macro avg      0.756     0.777     0.765       400
weighted avg      0.827     0.818     0.821       400

Confusion matrix:
 [[258  43]
 [ 30  69]]
ROC-AUC: 0.722


## 8) Modèle Avancé - CatBoost
CatBoost est particulièrement performant avec les données catégorielles.

In [14]:
cat_model = Pipeline([
    ('prep', preprocess),
    ('clf', CatBoostClassifier(iterations=100, learning_rate=0.05, depth=4, verbose=0, random_state=42, auto_class_weights='Balanced',l2_leaf_reg=3))
])

cat_model.fit(X_train, y_train)
pred_cat = cat_model.predict(X_test)
proba_cat = cat_model.predict_proba(X_test)[:, 1]

print('=== CatBoost ===')
print(classification_report(y_test, pred_cat, digits=3))
print('ROC-AUC:', round(roc_auc_score(y_test, proba_cat), 3))

=== CatBoost ===
              precision    recall  f1-score   support

           0      0.941     0.841     0.888       301
           1      0.634     0.838     0.722        99

    accuracy                          0.840       400
   macro avg      0.787     0.839     0.805       400
weighted avg      0.865     0.840     0.847       400

ROC-AUC: 0.851


## 11) Optimisation Avancée (Weight Balancing + Tuning)
Nous allons utiliser l'équilibrage des poids internes aux modèles et RandomizedSearchCV pour trouver les meilleurs paramètres.

In [15]:
# 11.1) XGBoost Tuned (Weight Balancing)
xgb_tuned_pipe = Pipeline([
    ('prep', preprocess),
    ('clf', XGBClassifier(
        random_state=42, 
        use_label_encoder=False, 
        eval_metric='logloss',
        scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum()
    ))
])

param_grid_xgb = {
    'clf__n_estimators': [100, 500, 1000],
    'clf__max_depth': [3, 5, 7, 9],
    'clf__learning_rate': [0.01, 0.05, 0.1, 0.2],
    'clf__subsample': [0.6, 0.8, 1.0],
    'clf__colsample_bytree': [0.6, 0.8, 1.0]
}

xgb_search = RandomizedSearchCV(
    xgb_tuned_pipe, param_distributions=param_grid_xgb, 
    n_iter=10, cv=3, scoring='roc_auc', n_jobs=-1, random_state=42, verbose=1
)

xgb_search.fit(X_train, y_train)
best_xgb = xgb_search.best_estimator_

print('Meilleurs paramètres XGB:', xgb_search.best_params_)
pred_xgb_tuned = best_xgb.predict(X_test)
proba_xgb_tuned = best_xgb.predict_proba(X_test)[:, 1]

print('\n=== XGBoost Tuned (Balanced Weights) ===')
print(classification_report(y_test, pred_xgb_tuned, digits=3))
print('ROC-AUC:', round(roc_auc_score(y_test, proba_xgb_tuned), 3))


Fitting 3 folds for each of 10 candidates, totalling 30 fits
Meilleurs paramètres XGB: {'clf__subsample': 0.8, 'clf__n_estimators': 500, 'clf__max_depth': 3, 'clf__learning_rate': 0.01, 'clf__colsample_bytree': 0.8}

=== XGBoost Tuned (Balanced Weights) ===
              precision    recall  f1-score   support

           0      0.948     0.841     0.891       301
           1      0.639     0.859     0.733        99

    accuracy                          0.845       400
   macro avg      0.793     0.850     0.812       400
weighted avg      0.871     0.845     0.852       400

ROC-AUC: 0.848


In [16]:
# 11.1b) CatBoost Tuned (Weight Balancing)
cat_tuned_pipe = Pipeline([ 
    ('prep', preprocess),
    ('clf', CatBoostClassifier(verbose=0, random_state=42, auto_class_weights='Balanced'))
])

param_grid_cat = {
    'clf__iterations': [100, 500, 1000],
    'clf__depth': [4, 6, 8],
    'clf__learning_rate': [0.01, 0.05, 0.1],
    'clf__l2_leaf_reg': [1, 3, 5, 7]
}

cat_search = RandomizedSearchCV(
    cat_tuned_pipe, param_distributions=param_grid_cat, 
    n_iter=5, cv=3, scoring='roc_auc', n_jobs=-1, random_state=42, verbose=1
)

cat_search.fit(X_train, y_train)
best_cat = cat_search.best_estimator_

print('Meilleurs paramètres CatBoost:', cat_search.best_params_)
pred_cat_tuned = best_cat.predict(X_test)
proba_cat_tuned = best_cat.predict_proba(X_test)[:, 1]

print('\n=== CatBoost Tuned (Balanced Weights) ===')
print(classification_report(y_test, pred_cat_tuned, digits=3))
print('ROC-AUC:', round(roc_auc_score(y_test, proba_cat_tuned), 3))


Fitting 3 folds for each of 5 candidates, totalling 15 fits
Meilleurs paramètres CatBoost: {'clf__learning_rate': 0.05, 'clf__l2_leaf_reg': 3, 'clf__iterations': 100, 'clf__depth': 4}

=== CatBoost Tuned (Balanced Weights) ===
              precision    recall  f1-score   support

           0      0.941     0.841     0.888       301
           1      0.634     0.838     0.722        99

    accuracy                          0.840       400
   macro avg      0.787     0.839     0.805       400
weighted avg      0.865     0.840     0.847       400

ROC-AUC: 0.851


In [17]:
# 11.2) Optimisation du Seuil (Threshold Tuning)
from sklearn.metrics import precision_recall_curve

precisions, recalls, thresholds = precision_recall_curve(y_test, proba_xgb_tuned)

# Trouver le seuil qui maximise le F1-score
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-10)
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]

print(f'Meilleur seuil (F1-score): {best_threshold:.3f}')

# Nouvelle prédiction avec le seuil optimisé
pred_custom = (proba_xgb_tuned >= best_threshold).astype(int)

print('\n=== XGBoost Tuned (Seuil Optimisé) ===')
print(classification_report(y_test, pred_custom, digits=3))

Meilleur seuil (F1-score): 0.540

=== XGBoost Tuned (Seuil Optimisé) ===
              precision    recall  f1-score   support

           0      0.948     0.841     0.891       301
           1      0.639     0.859     0.733        99

    accuracy                          0.845       400
   macro avg      0.793     0.850     0.812       400
weighted avg      0.871     0.845     0.852       400



Comparaison entre le xgboost avant le tuning et apres

In [14]:
import joblib

# Sauvegarde du meilleur modèle XGBoost
joblib.dump(best_xgb, 'xgb_best_model.joblib')

# Sauvegarde du meilleur modèle CatBoost
joblib.dump(best_cat, 'cat_best_model.joblib')

print("✅ Modèles sauvegardés avec succès !")

✅ Modèles sauvegardés avec succès !


## 13) Export des Modèles
Cette section sauvegarde les modèles entraînés pour qu'ils soient utilisables par l'interface Streamlit.

In [15]:
# 13) Sauvegarde des modèles pour l'application Streamlit
import joblib
import os

# On sauvegarde les meilleurs modèles trouvés par RandomizedSearchCV
# Note: ces fichiers seront créés à la racine du projet pour être lus par app.py
if 'best_xgb' in globals():
    joblib.dump(best_xgb, '../xgb_best_model.joblib')
    print("Modèle XGBoost sauvegardé sous 'xgb_best_model.joblib'")
else:
    print("Erreur: best_xgb non trouvé. Veuillez lancer l'entraînement XGBoost.")

if 'best_cat' in globals():
    joblib.dump(best_cat, '../cat_best_model.joblib')
    print("Modèle CatBoost sauvegardé sous 'cat_best_model.joblib'")
else:
    print("Erreur: best_cat non trouvé. Veuillez lancer l'entraînement CatBoost.")


Modèle XGBoost sauvegardé sous 'xgb_best_model.joblib'
Modèle CatBoost sauvegardé sous 'cat_best_model.joblib'


## 15) Comparaison Sans SMOTE vs Avec SMOTE

Objectif: comparer precision/recall/AUC globaux et recall par categorie `incident_severity`.


In [16]:
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

# Reutilise X_train, X_test, y_train, y_test, preprocess definis plus haut
xgb_base_cmp = XGBClassifier(
    n_estimators=300,
    max_depth=3,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss',
    random_state=42
)

# 1) Sans SMOTE
pipe_no_smote = Pipeline([
    ('prep', preprocess),
    ('clf', xgb_base_cmp)
])
pipe_no_smote.fit(X_train, y_train)
proba_no = pipe_no_smote.predict_proba(X_test)[:, 1]
pred_no = (proba_no >= 0.5).astype(int)

# 2) Avec SMOTE
pipe_smote = ImbPipeline([
    ('prep', preprocess),
    ('smote', SMOTE(random_state=42)),
    ('clf', xgb_base_cmp)
])
pipe_smote.fit(X_train, y_train)
proba_sm = pipe_smote.predict_proba(X_test)[:, 1]
pred_sm = (proba_sm >= 0.5).astype(int)

def global_metrics(y_true, pred, proba):
    return {
        "precision": round(precision_score(y_true, pred), 3),
        "recall": round(recall_score(y_true, pred), 3),
        "f1": round(f1_score(y_true, pred), 3),
        "auc": round(roc_auc_score(y_true, proba), 3),
    }

m_no = global_metrics(y_test, pred_no, proba_no)
m_sm = global_metrics(y_test, pred_sm, proba_sm)
print('=== Global Metrics ===')
print('Sans SMOTE:', m_no)
print('Avec SMOTE:', m_sm)

# Recall par categorie de gravite
sev_test = X_test['incident_severity'].reset_index(drop=True)
tmp = pd.DataFrame({
    "incident_severity": sev_test,
    "y_true": y_test.reset_index(drop=True),
    "pred_no_smote": pred_no,
    "pred_smote": pred_sm
})

def recall_grp(df, pred_col):
    out = {}
    for g, d in df.groupby('incident_severity'):
        pos = d[d['y_true'] == 1]
        if len(pos) == 0:
            out[g] = None
        else:
            out[g] = round((pos[pred_col] == 1).mean(), 3)
    return out

print('\n=== Recall par incident_severity (classe fraude=1) ===')
print('Sans SMOTE:', recall_grp(tmp, "pred_no_smote"))
print('Avec SMOTE:', recall_grp(tmp, "pred_smote"))


=== Global Metrics ===
Sans SMOTE: {'precision': 0.636, 'recall': 0.636, 'f1': 0.636, 'auc': 0.847}
Avec SMOTE: {'precision': 0.645, 'recall': 0.717, 'f1': 0.679, 'auc': 0.854}

=== Recall par incident_severity (classe fraude=1) ===
Sans SMOTE: {'MAJOR DAMAGE': np.float64(0.701), 'MINOR DAMAGE': np.float64(0.421), 'TOTAL LOSS': np.float64(0.545), 'TRIVIAL DAMAGE': np.float64(1.0)}
Avec SMOTE: {'MAJOR DAMAGE': np.float64(0.806), 'MINOR DAMAGE': np.float64(0.474), 'TOTAL LOSS': np.float64(0.545), 'TRIVIAL DAMAGE': np.float64(1.0)}


## 16) Diagnostic Erreurs (Faux Positifs / Faux N?gatifs)

Analyse des cas non logiques: top faux positifs, top faux n?gatifs, et taux d'erreur par `incident_severity`.


In [17]:
# Pr?requis: utiliser un mod?le d?j? entra?n? dans le notebook
# On essaie par priorit?: best_xgb, puis xgb_model
model_for_diag = None
for m in ['best_xgb', 'xgb_model', 'xgb_v2']:
    if m in globals():
        model_for_diag = globals()[m]
        print(f"Mod?le utilis? pour diagnostic: {m}")
        break

if model_for_diag is None:
    raise ValueError("Aucun mod?le trouv?. Ex?cutez d'abord une cellule d'entra?nement.")

# V?rifie pr?sence du split standard
required = ['X_test', 'y_test']
for v in required:
    if v not in globals():
        raise ValueError(f"Variable manquante: {v}. Ex?cutez la section split avant.")

proba_diag = model_for_diag.predict_proba(X_test)[:, 1]
thr_diag = 0.5
pred_diag = (proba_diag >= thr_diag).astype(int)

diag = X_test.copy().reset_index(drop=True)
diag['y_true'] = y_test.reset_index(drop=True)
diag['proba'] = proba_diag
diag['y_pred'] = pred_diag

# Faux positifs: pr?dits fraude mais vrais non-fraude
fp = diag[(diag['y_true']==0) & (diag['y_pred']==1)].copy()
# Faux n?gatifs: pr?dits non-fraude mais vrais fraude
fn = diag[(diag['y_true']==1) & (diag['y_pred']==0)].copy()

# Tri par confiance du mod?le
fp_top = fp.sort_values('proba', ascending=False).head(15)
fn_top = fn.sort_values('proba', ascending=True).head(15)

cols_show = [c for c in [
    "incident_severity", "incident_type", "police_report_available",
    "total_claim_amount", "policy_annual_premium", "policy_deductable",
    "witnesses", "bodily_injuries", "months_as_customer", "claim_delay_days",
    "y_true", "y_pred", "proba"
] if c in diag.columns]

print("\n=== Top Faux Positifs (non-fraude class?e fraude) ===")
display(fp_top[cols_show])

print("\n=== Top Faux N?gatifs (fraude class?e non-fraude) ===")
display(fn_top[cols_show])

# Taux d'erreur par gravit?
if "incident_severity" in diag.columns:
    seg = diag.copy()
    seg['is_error'] = (seg['y_true'] != seg['y_pred']).astype(int)
    seg_summary = seg.groupby('incident_severity').agg(
        n=('is_error', 'size'),
        error_rate=('is_error', 'mean'),
        avg_proba=('proba', 'mean'),
        fraud_rate_true=('y_true', 'mean')
    ).sort_values('error_rate', ascending=False)
    print("\n=== Erreurs par incident_severity ===")
    display(seg_summary.round(3))


Mod?le utilis? pour diagnostic: best_xgb

=== Top Faux Positifs (non-fraude class?e fraude) ===


,incident_severity,incident_type,police_report_available,total_claim_amount,policy_annual_premium,policy_deductable,witnesses,bodily_injuries,months_as_customer,claim_delay_days,y_true,y_pred,proba
342,MAJOR DAMAGE,SINGLE VEHICLE COLLISION,YES,60600,1459.99,500,2,1,381,8675,0,1,0.896587
50,MAJOR DAMAGE,MULTI-VEHICLE COLLISION,NO,56430,1082.49,500,3,2,439,2291,0,1,0.891166
96,MAJOR DAMAGE,SINGLE VEHICLE COLLISION,NO,59940,1276.43,2000,3,0,146,7735,0,1,0.883543
37,MAJOR DAMAGE,SINGLE VEHICLE COLLISION,NO,59400,1411.43,500,1,2,328,3324,0,1,0.867381
309,MAJOR DAMAGE,MULTI-VEHICLE COLLISION,YES,72800,979.26,2000,2,0,140,5343,0,1,0.866275
383,MAJOR DAMAGE,MULTI-VEHICLE COLLISION,NO,75600,954.16,1000,0,0,64,5448,0,1,0.863989
191,MAJOR DAMAGE,SINGLE VEHICLE COLLISION,NO,59500,1625.65,1000,3,1,330,6091,0,1,0.863426
65,MAJOR DAMAGE,MULTI-VEHICLE COLLISION,YES,72000,1301.72,500,3,0,124,7828,0,1,0.860366
274,MINOR DAMAGE,SINGLE VEHICLE COLLISION,YES,83490,1578.54,2000,3,1,266,2169,0,1,0.858580
232,MAJOR DAMAGE,SINGLE VEHICLE COLLISION,NO,56600,1568.47,2000,0,2,46,5825,0,1,0.854957



=== Top Faux N?gatifs (fraude class?e non-fraude) ===


,incident_severity,incident_type,police_report_available,total_claim_amount,policy_annual_premium,policy_deductable,witnesses,bodily_injuries,months_as_customer,claim_delay_days,y_true,y_pred,proba
311,MINOR DAMAGE,PARKED CAR,NO,6120,1554.86,2000,2,1,140,6500,1,0,0.073573
150,MINOR DAMAGE,MULTI-VEHICLE COLLISION,YES,53400,1090.65,500,2,1,156,8011,1,0,0.080460
46,MINOR DAMAGE,PARKED CAR,YES,5580,994.74,1000,2,2,291,3209,1,0,0.084069
392,MINOR DAMAGE,SINGLE VEHICLE COLLISION,NO,77000,944.03,500,3,2,232,660,1,0,0.089727
99,TOTAL LOSS,MULTI-VEHICLE COLLISION,NO,47700,1110.37,1000,1,0,258,5800,1,0,0.106815
212,MINOR DAMAGE,MULTI-VEHICLE COLLISION,NO,54240,1053.04,1000,3,1,101,976,1,0,0.112231
368,MINOR DAMAGE,SINGLE VEHICLE COLLISION,NO,89700,1239.22,500,1,0,107,1262,1,0,0.124751
145,MINOR DAMAGE,SINGLE VEHICLE COLLISION,NO,76230,1340.71,500,2,2,271,2524,1,0,0.125424
398,TOTAL LOSS,MULTI-VEHICLE COLLISION,YES,63570,1158.03,500,3,1,104,6387,1,0,0.130306
175,MINOR DAMAGE,SINGLE VEHICLE COLLISION,NO,48950,1185.78,1000,2,0,10,180,1,0,0.143610



=== Erreurs par incident_severity ===


,n,error_rate,avg_proba,fraud_rate_true
incident_severity,,,,
MAJOR DAMAGE,111,0.396,0.770,0.604
MINOR DAMAGE,142,0.077,0.164,0.134
TOTAL LOSS,111,0.063,0.205,0.099
TRIVIAL DAMAGE,36,0.000,0.156,0.056


## 17) D?cision avec Seuils par Gravit?

On applique un seuil plus strict pour `MAJOR DAMAGE` afin de r?duire les faux positifs sur ce segment.


In [18]:
# Pr?requis: variables de la section diagnostic (model_for_diag, X_test, y_test, proba_diag)
if "proba_diag" not in globals():
    proba_diag = model_for_diag.predict_proba(X_test)[:, 1]

# Seuils par segment (? ajuster)
severity_thresholds = {
    "MAJOR DAMAGE": 0.80,
    "MINOR DAMAGE": 0.50,
    "TOTAL LOSS": 0.50,
    "TRIVIAL DAMAGE": 0.50
}

sev = X_test["incident_severity"].reset_index(drop=True)
thr_row = sev.map(severity_thresholds).fillna(0.50).values
pred_seg = (proba_diag >= thr_row).astype(int)

from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score, f1_score

y_true_seg = y_test.reset_index(drop=True).values
print("=== Global metrics (seuils par gravit?) ===")
print("precision:", round(precision_score(y_true_seg, pred_seg), 3))
print("recall:", round(recall_score(y_true_seg, pred_seg), 3))
print("f1:", round(f1_score(y_true_seg, pred_seg), 3))
print("\nConfusion matrix:")
print(confusion_matrix(y_true_seg, pred_seg))
print("\nClassification report:")
print(classification_report(y_true_seg, pred_seg, digits=3))

seg_eval = pd.DataFrame({
    "incident_severity": sev,
    "y_true": y_true_seg,
    "y_pred_seg": pred_seg,
    "proba": proba_diag
})
seg_eval["is_error"] = (seg_eval["y_true"] != seg_eval["y_pred_seg"]).astype(int)

summary = seg_eval.groupby("incident_severity").agg(
    n=("is_error", "size"),
    error_rate=("is_error", "mean"),
    fraud_rate_true=("y_true", "mean"),
    predicted_fraud_rate=("y_pred_seg", "mean"),
    avg_proba=("proba", "mean")
).sort_values("error_rate", ascending=False)

print("\n=== Erreurs par gravit? (seuils segment?s) ===")
display(summary.round(3))


=== Global metrics (seuils par gravit?) ===
precision: 0.652
recall: 0.434
f1: 0.521

Confusion matrix:
[[278  23]
 [ 56  43]]

Classification report:
              precision    recall  f1-score   support

           0      0.832     0.924     0.876       301
           1      0.652     0.434     0.521        99

    accuracy                          0.802       400
   macro avg      0.742     0.679     0.698       400
weighted avg      0.788     0.802     0.788       400


=== Erreurs par gravit? (seuils segment?s) ===


,n,error_rate,fraud_rate_true,predicted_fraud_rate,avg_proba
incident_severity,,,,,
MAJOR DAMAGE,111,0.550,0.604,0.396,0.770
MINOR DAMAGE,142,0.077,0.134,0.085,0.164
TOTAL LOSS,111,0.063,0.099,0.072,0.205
TRIVIAL DAMAGE,36,0.000,0.056,0.056,0.156
